In [1]:
import pickle
from pathlib import Path

import decoupler as dc
import numpy as np
import pandas as pd

# 项目路径（相对 notebook 所在目录）
ROOT = Path("..")
GMT_DIR = ROOT / "data_interim" / "gmt"
OUT_DIR = ROOT / "results" / "enrichment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 分析参数
MIN_SIZE, MAX_SIZE = 15, 500   
SIG_PADJ = 0.05                # 显著性阈值（BH 校正后）
SHARED_FRAC = 0.7              # 一个通路在 ≥70% 的组织中激活 -> 共享模块

SEED = 2026
np.random.seed(SEED)
print(f"decoupler {dc.__version__}")

C:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


decoupler 2.2.0


In [2]:
GMT_FILES = {
    "hallmark": ("mh.all.v2026.1.Mm.symbols.gmt", None),
    "go_biological_process": ("m5.go.bp.v2026.1.Mm.symbols.gmt", None),
    "reactome_pathways": ("m2.cp.v2026.1.Mm.symbols.gmt", "REACTOME_"),
}

nets = {}
for coll, (fname, prefix) in GMT_FILES.items():
    net = dc.pp.read_gmt(GMT_DIR / fname)
    if prefix: 
        net = net[net["source"].str.startswith(prefix)]
    # 规模过滤
    size = net.groupby("source").size()
    keep = size[(size >= MIN_SIZE) & (size <= MAX_SIZE)].index
    nets[coll] = net[net["source"].isin(keep)].copy()
    print(f"{coll:24s} pathways={nets[coll]['source'].nunique():5d}  "
          f"genes={nets[coll]['target'].nunique():6d}")

print(f"\ntotal pathways: {sum(n['source'].nunique() for n in nets.values())}")

hallmark                 pathways=   50  genes=  4291


go_biological_process    pathways= 4323  genes= 16497
reactome_pathways        pathways=  836  genes=  8898

total pathways: 5209


In [3]:
# 按教程约定：取 stat 列（t-value），转置为「contrast × gene」宽矩阵，
# 每行是一个「组织 × 细胞类型」节点，列名为基因。
# 注意：各节点基因集不完全一致，取所有节点的基因交集，避免引入 NaN
with open(ROOT / "all_results.pkl", "rb") as f:
    results = pickle.load(f)

rows = {}
for (tissue, celltype), res_df in results.items():
    node = f"{tissue}__{celltype}"
    stat = res_df.set_index("variable")["stat"]
    rows[node] = stat.replace([np.inf, -np.inf], np.nan).dropna()

data = pd.DataFrame(rows).T          # contrasts × genes（各节点基因并集）
n_all = data.shape[1]
data = data.dropna(axis=1, how="any")   # 取交集，保证无 NaN
print(f"contrast × gene matrix: {data.shape}  (shared genes {data.shape[1]}/{n_all})")
print(f"nodes: {list(data.index)}")

contrast × gene matrix: (8, 9447)  (shared genes 9447/9560)
nodes: ['Kidney__lymphoid', 'Kidney__myeloid/macrophage', 'Liver__lymphoid', 'Liver__myeloid/macrophage', 'Lung__lymphoid', 'Lung__myeloid/macrophage', 'spleen/marrow__lymphoid', 'spleen/marrow__myeloid/macrophage']


In [4]:
# 一次传入全部 8 个 contrast，ULM 为每个「节点 × 通路」返回一个分数与校正 p 值。
# dc.pp.prune 先把网络裁剪到与统计量矩阵共享的基因，剔除无法打分的基因集。
# 每个通路的来源集合单独记录在 coll_of 中，供长表标注使用。
acts, pads, coll_of = [], [], {}

for coll, net in nets.items():
    net_p = dc.pp.prune(data.columns, net, tmin=MIN_SIZE)
    a, p = dc.mt.ulm(data=data, net=net_p)
    acts.append(a)
    pads.append(p)
    for gene_set in a.columns:
        coll_of[gene_set] = coll
    print(f"{coll:24s} scored={a.shape[1]:5d}  significant(node×pathway)={int((p < SIG_PADJ).sum().sum())}")

# 合并三类集合：行 = 节点（组织__细胞类型），列 = 通路
score_mat = pd.concat(acts, axis=1)
padj_mat = pd.concat(pads, axis=1)
assert score_mat.columns.is_unique, "通路名跨集合重复，需加集合命名空间"
print(f"\ncombined score matrix: {score_mat.shape}")

hallmark                 scored=   49  significant(node×pathway)=140


go_biological_process    scored= 2913  significant(node×pathway)=2213
reactome_pathways        scored=  595  significant(node×pathway)=766

combined score matrix: (8, 3557)


In [6]:
# ===================== 5. 组织 × 细胞类型 × 通路 三维矩阵 =====================
# 长表形式（每行一个 组织 × 细胞类型 × 通路 单元），便于下游筛选与汇总；
# 宽表（行 = 节点，列 = 通路）供热图、聚类等下游直接读取。
index = pd.MultiIndex.from_tuples(
    [n.split("__") for n in score_mat.index], names=["tissue", "celltype"]
)

def to_long(mat, value_name):
    out = mat.copy()
    out.index = index
    return out.stack().rename(value_name).reset_index().rename(columns={"level_2": "pathway"})

cube = to_long(score_mat, "score").merge(
    to_long(padj_mat, "padj"), on=["tissue", "celltype", "pathway"], how="left"
)
cube["collection"] = cube["pathway"].map(coll_of)
cube["significant"] = cube["padj"] < SIG_PADJ

print(f"3D cube (tissue × celltype × pathway): {cube.shape}")
print(cube.head().to_string(index=False))

3D cube (tissue × celltype × pathway): (28456, 7)
tissue celltype                      pathway     score     padj collection  significant
Kidney lymphoid        HALLMARK_ADIPOGENESIS -0.957775 0.410882   hallmark        False
Kidney lymphoid HALLMARK_ALLOGRAFT_REJECTION -0.978361 0.410882   hallmark        False
Kidney lymphoid   HALLMARK_ANDROGEN_RESPONSE -1.754578 0.169080   hallmark        False
Kidney lymphoid        HALLMARK_ANGIOGENESIS -0.724359 0.547007   hallmark        False
Kidney lymphoid     HALLMARK_APICAL_JUNCTION -4.520449 0.000061   hallmark         True


In [7]:
# ===================== 6. 共享模块 / 组织特异模块识别 =====================
# 判定口径（沿用项目既有标准，并加入方向一致性约束）：
#   1) 一个「通路 × 节点」显著 = padj<0.05，方向由 score 的符号决定；
#   2) 逐方向统计激活组织数：仅当同一通路在多个组织中**方向一致**时
#      才计为共享。同一通路若在不同组织中方向相反（up/down 混合），
#      则不计入该方向的共享模块（避免把「此消彼长」误判为共享）。
#   3) 模块类型（以主导方向的激活组织数 n 判定）：
#        Shared   —— n ≥ 70% 的组织
#        Specific —— n == 1
#        Partial  —— 其余
n_tissues = cube["tissue"].nunique()
print(f"tissues in analysis: {n_tissues}")

# 每个「通路 × 方向」激活的组织集合
sig = cube[cube["significant"]].copy()
sig["direction"] = np.where(sig["score"] > 0, "up_in_old", "down_in_old")

recs = []
for (pw, direction), sub in sig.groupby(["pathway", "direction"]):
    tis = sorted(sub["tissue"].unique())
    n = len(tis)
    mtype = ("Shared" if n >= n_tissues * SHARED_FRAC
             else "Specific" if n == 1 else "Partial")
    recs.append({
        "pathway": pw,
        "collection": sub["collection"].iloc[0],
        "direction": direction,
        "n_tissues": n,
        "active_tissues": "|".join(tis),
        "mean_score": sub["score"].mean(),
        "n_nodes": len(sub),
        "module_type": mtype,
    })

mod = pd.DataFrame(recs).sort_values(
    ["module_type", "n_tissues", "pathway"], ascending=[True, False, True]
)

print("\nmodule type distribution（以 通路×方向 为单位）:")
print(mod["module_type"].value_counts().to_string())
print(f"\nrows: {len(mod)}  (unique pathways: {mod['pathway'].nunique()})")

tissues in analysis: 4

module type distribution（以 通路×方向 为单位）:
module_type
Specific    1392
Partial      487
Shared       118

rows: 1997  (unique pathways: 1705)


In [8]:
for mtype in ["Shared", "Specific"]:
    sub = mod[mod["module_type"] == mtype]
    print(f"\n===== {mtype} ({len(sub)}) =====")
    if mtype == "Shared":
        # 共享模块：按平均效应量绝对值排序，展示最强的一批
        top = sub.reindex(sub["mean_score"].abs().sort_values(ascending=False).index).head(12)
        print(top[["pathway", "collection", "direction", "n_tissues",
                   "active_tissues", "mean_score"]].to_string(index=False))
    else:
        print(sub[["pathway", "collection", "direction", "active_tissues",
                   "mean_score"]].head(12).to_string(index=False))


===== Shared (118) =====
                                                                                                              pathway            collection direction  n_tissues                  active_tissues  mean_score
                                    REACTOME_NONSENSE_MEDIATED_DECAY_NMD_INDEPENDENT_OF_THE_EXON_JUNCTION_COMPLEX_EJC     reactome_pathways up_in_old          3               Kidney|Liver|Lung    9.567875
REACTOME_ZNF598_AND_THE_RIBOSOME_ASSOCIATED_QUALITY_TRIGGER_RQT_COMPLEX_DISSOCIATE_A_RIBOSOME_STALLED_ON_A_NO_GO_MRNA     reactome_pathways up_in_old          3               Kidney|Liver|Lung    9.483240
                                                                           REACTOME_EUKARYOTIC_TRANSLATION_INITIATION     reactome_pathways up_in_old          3               Kidney|Liver|Lung    9.398054
                                                                                 REACTOME_NONSENSE_MEDIATED_DECAY_NMD     reactome_pathways up_in_old     

In [9]:
# ===================== 8. 冗余通路聚类 -> 上位主题归并 =====================
# 目的：GO:BP / Reactome 中大量通路高度冗余（同一生物学过程被拆成几十条）。
#       这里按「通路在 8 个节点上的打分谱」做层次聚类，把相似通路并成一个「上位主题」，
#       便于解释与画图。**原始打分与逐通路模块分类完整保留**（见 Cell 9 存盘）。
# 口径：
#   - 仅在「集合 × 方向」内部聚类：上调通路与下调通路不合并，避免主题语义混杂；
#   - 相似度 = 通路打分向量在节点维度上的 Pearson 相关（r）；
#     距离 = 1 - r，average linkage，切在 r > CLUSTER_R（即距离 < 1 - CLUSTER_R）；
#   - 代表通路 = 该主题内各节点 |score| 均值最大者（效应最强、最能代表主题）。
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

CLUSTER_R = 0.7      # 相关性阈值：r > 0.7 视为冗余（阈值越松归并越狠，越严保留越多细节）

theme_rows = []      # 每个聚类一行的汇总
member_rows = []     # 逐通路的归属映射（用于下游回连）

for (coll, direction), grp in mod.groupby(["collection", "direction"]):
    terms = [t for t in grp["pathway"] if t in score_mat.columns]
    if len(terms) < 3:            # 少于 3 条无聚类意义
        continue

    # 打分谱：行 = 通路，列 = 节点（score_mat 是 节点 × 通路，故需转置）
    sub = score_mat[terms].T.dropna(axis=0, how="any")
    if sub.shape[0] < 3:
        continue

    # 通路 × 通路 相关 -> 距离矩阵
    corr = sub.T.corr().fillna(0)
    dist = 1 - corr
    np.fill_diagonal(dist.values, 0)
    condensed = squareform(dist.values, checks=False)
    if not np.isfinite(condensed).all():
        continue

    labels = fcluster(linkage(condensed, method="average"),
                      t=1 - CLUSTER_R, criterion="distance")

    for lab in np.unique(labels):
        members = sub.index[labels == lab].tolist()
        # 代表通路：各节点 |score| 均值最大
        rep = sub.loc[members].abs().mean(axis=1).idxmax()
        theme_id = f"{coll}|{direction}|{lab}"
        theme_rows.append({
            "theme_id": theme_id,
            "collection": coll,
            "direction": direction,
            "representative": rep,
            "n_pathways": len(members),
            "members": "|".join(members),
            "sum_abs_score": float(sub.loc[members].abs().sum().sum()),
        })
        for p in members:
            member_rows.append({
                "pathway": p, "theme_id": theme_id, "representative": rep,
                "collection": coll, "direction": direction,
                "is_representative": p == rep,
            })

themes = pd.DataFrame(theme_rows).sort_values("sum_abs_score", ascending=False)
pathway_to_theme = pd.DataFrame(member_rows)
# 仅含 ≥2 条通路的主题 = 真正发生了「归并」的主题
themes_merged = themes[themes["n_pathways"] >= 2].reset_index(drop=True)

print(f"参与聚类的「通路×方向」条目: {len(mod)}")
print(f"归并出的上位主题（≥2 条通路）: {len(themes_merged)}"
      f"  覆盖唯一通路: {themes_merged['members'].str.split('|').explode().nunique()}")

print("\n各集合 × 方向的归并情况（通路数 -> 主题数）:")
summary = (themes.groupby(["collection", "direction"])
           .agg(n_pathways=("n_pathways", "sum"), n_themes=("theme_id", "count")))
summary["compression"] = (summary["n_pathways"] / summary["n_themes"]).round(1)
print(summary.to_string())

print("\n===== 覆盖通路最多的 12 个上位主题 =====")
disp = themes_merged.head(12).copy()
disp["representative"] = disp["representative"].str.replace(
    r"^(GOBP_|REACTOME_|HALLMARK_)", "", regex=True).str.slice(0, 60)
print(disp[["collection", "direction", "representative", "n_pathways"]].to_string(index=False))

参与聚类的「通路×方向」条目: 1997
归并出的上位主题（≥2 条通路）: 133  覆盖唯一通路: 1684

各集合 × 方向的归并情况（通路数 -> 主题数）:
                                   n_pathways  n_themes  compression
collection            direction                                     
go_biological_process down_in_old         632        36         17.6
                      up_in_old           868        49         17.7
hallmark              down_in_old          27        10          2.7
                      up_in_old            35        12          2.9
reactome_pathways     down_in_old         184        24          7.7
                      up_in_old           251        28          9.0

===== 覆盖通路最多的 12 个上位主题 =====
           collection   direction                                               representative  n_pathways
go_biological_process down_in_old                                          RIBOSOME_BIOGENESIS         118
go_biological_process   up_in_old                                DEFENSE_RESPONSE_TO_BACTERIUM         134
    reactome

In [ ]:
# ===================== 9. 冗余通路归并（基因集重叠 + 强制家族） =====================
# 【第一步】基因集 Jaccard 自动归并（**同方向内**）
#   sim = |A ∩ B| / |A ∪ B|；仅在同一 direction 内比较，sim >= JACCARD_THR 则合并。
#   这一步处理「基因集本身高度重叠」的冗余（如 GO:BP 内部 4 条电子传递链通路）。
# 【第二步】强制家族归并（**跨方向、跨集合**）
#   有些通路基因集重叠不高（因不同数据库收录的成员基因范围不同），
#   但在 8 个节点上的**打分谱完全一致**（实测符号一致率 8/8），
#   生物学上确属同一过程。这类由人工审核后写入 FORCED_FAMILIES 强制合并。
#
#   判定依据（以「氧化磷酸化」家族为例，实测）：
#     - 11 条成员在 8 节点上的**打分符号两两一致率 = 8/8（全部）**
#     - 其中 10 对的「同号且同时显著」节点数 = 6/8
#     - 列方向一致：Kidney髓 / spleen髓 全正，Liver髓 / Lung淋 全负
#   → 即热图上这几行确实「长得一样」，应合并为一行。
#
# 交叉验证 = decoupler 的 net_corr（dc.pp.net_corr）：
#   计算通路两两的打分相关性（corr / pval / padj），是独立于 Jaccard 的度量。
#
# 代表通路：组内取 |mean_score| 最大者。强制家族另记 direction = "mixed"（若跨方向）。

JACCARD_THR = 0.5     # 基因集 Jaccard 阈值：>=0.5 视为同一过程的不同命名
VERIFY_CORR = 0.5     # net_corr 交叉验证阈值

# ---------- 强制家族名单（人工审核，跨方向跨集合并） ----------
# 定义口径：基因集可能不同（跨数据库收录范围差异），但**打分谱一致**
FORCED_FAMILIES = {
    # 氧化磷酸化 / 电子传递链（跨 GOBP + Reactome + Hallmark）
    "OXPHOS": [
        "GOBP_OXIDATIVE_PHOSPHORYLATION",
        "GOBP_AEROBIC_ELECTRON_TRANSPORT_CHAIN",
        "GOBP_ATP_SYNTHESIS_COUPLED_ELECTRON_TRANSPORT",
        "GOBP_ELECTRON_TRANSPORT_CHAIN",
        "GOBP_PROTON_MOTIVE_FORCE_DRIVEN_ATP_SYNTHESIS",
        "GOBP_MITOCHONDRIAL_ELECTRON_TRANSPORT_NADH_TO_UBIQUINONE",
        "REACTOME_RESPIRATORY_ELECTRON_TRANSPORT",
        "REACTOME_AEROBIC_RESPIRATION_AND_RESPIRATORY_ELECTRON_TRANSPORT",
        "REACTOME_COMPLEX_I_BIOGENESIS",
        "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    ],
    "COMPLEMENT": [
        "GOBP_COMPLEMENT_ACTIVATION",
        "GOBP_REGULATION_OF_COMPLEMENT_ACTIVATION",
        "REACTOME_COMPLEMENT_CASCADE",
    ],
    "STEROID_METABOLISM": [
        "GOBP_STEROID_METABOLIC_PROCESS",
        "GOBP_STEROL_BIOSYNTHETIC_PROCESS",
        "REACTOME_METABOLISM_OF_STEROIDS",
    ],
    "HEME_PORPHYRIN": [
        "HALLMARK_HEME_METABOLISM",
        "GOBP_PORPHYRIN_CONTAINING_COMPOUND_METABOLIC_PROCESS",
        "GOBP_HEME_METABOLIC_PROCESS",
    ],
    "LEUKOCYTE_MIGRATION": [
        "GOBP_LEUKOCYTE_MIGRATION",
        "GOBP_MONOCYTE_CHEMOTAXIS",
        "GOBP_POSITIVE_CHEMOTAXIS",
        "GOBP_LEUKOCYTE_MIGRATION_INVOLVED_IN_INFLAMMATORY_RESPONSE",
        "REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES",
    ],
    "MITOSIS_CELL_CYCLE": [
        "REACTOME_THE_ROLE_OF_GTSE1_IN_G2_M_PROGRESSION_AFTER_G2_CHECKPOINT",
        "REACTOME_MITOTIC_METAPHASE_AND_ANAPHASE",
        "REACTOME_MITOTIC_G2_G2_M_PHASES",
    ],
    "TRANSLATION_RRNA": [
        "GOBP_CYTOPLASMIC_TRANSLATION",
        "REACTOME_MAJOR_PATHWAY_OF_RRNA_PROCESSING_IN_THE_NUCLEOLUS_AND_CYTOSOL",
    ],
    "NMD_TRANSLATION_QC": [
        "REACTOME_NONSENSE_MEDIATED_DECAY_NMD_INDEPENDENT_OF_THE_EXON_JUNCTION_COMPLEX_EJC",
        "GOBP_TRANSLATION_AT_SYNAPSE",
    ],
    "ANTIGEN_PRESENTATION": [
        "GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EXOGENOUS_PEPTIDE_ANTIGEN",
        "GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_PEPTIDE_ANTIGEN",
    ],
    "INFLAMMATORY_RESPONSE": [
        "GOBP_POSITIVE_REGULATION_OF_INFLAMMATORY_RESPONSE",
        "GOBP_REGULATION_OF_INFLAMMATORY_RESPONSE",
    ],
    "ROS_PRODUCTION": [
        "REACTOME_ROS_AND_RNS_PRODUCTION_IN_PHAGOCYTES",
        "GOBP_SUPEROXIDE_METABOLIC_PROCESS",
    ],
    "ANTIMICROBIAL_PEPTIDE": [
        "GOBP_ANTIMICROBIAL_HUMORAL_IMMUNE_RESPONSE_MEDIATED_BY_ANTIMICROBIAL_PEPTIDE",
    ],
    "BACTERIAL_DEFENSE": [
        "GOBP_DEFENSE_RESPONSE_TO_BACTERIUM",
        "GOBP_RESPONSE_TO_MOLECULE_OF_BACTERIAL_ORIGIN",
        "GOBP_DEFENSE_RESPONSE_TO_GRAM_NEGATIVE_BACTERIUM",
        "GOBP_ANTIMICROBIAL_HUMORAL_IMMUNE_RESPONSE_MEDIATED_BY_ANTIMICROBIAL_PEPTIDE",
    ],
    "CHEMICAL_STRESS_RESPONSE": [
        "REACTOME_CELLULAR_RESPONSE_TO_CHEMICAL_STRESS",
        "GOBP_RESPONSE_TO_TOXIC_SUBSTANCE",
    ],
    "INFLAMMATORY_RESPONSE": [
        "GOBP_POSITIVE_REGULATION_OF_INFLAMMATORY_RESPONSE",
        "GOBP_REGULATION_OF_INFLAMMATORY_RESPONSE",
        "GOBP_INFLAMMATORY_RESPONSE_TO_ANTIGENIC_STIMULUS",
    ],
}
NOT_MERGED = {
    "G9_ antibacterial_defense": [
    ],
    "G10_T_cell_vs_PD1": [
        "GOBP_REGULATION_OF_T_CELL_ACTIVATION",
        "REACTOME_CO_INHIBITION_BY_PD_1",
    ],
    "G6_respiratory_burst": [
        "GOBP_RESPIRATORY_BURST",
    ],
    "G3_antigen_reactome": [
        "REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION",
    ],
}

genesets = {}
for _coll, _net in nets.items():
    for _src, _sub in _net.groupby("source"):
        genesets[_src] = set(_sub["target"])

def _jaccard(a, b):
    A, B = genesets.get(a, set()), genesets.get(b, set())
    if not A or not B:
        return np.nan
    return len(A & B) / len(A | B)

def _merge_by_similarity(terms, sim_fn, thr):
    """并查集：相似度 >= thr 的通路连边，返回 {root: [members]}"""
    parent = {t: t for t in terms}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, a in enumerate(terms):
        for b in terms[i + 1:]:
            try:
                s = sim_fn(a, b)
            except Exception:
                continue
            if s is not None and np.isfinite(s) and s >= thr:
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[rb] = ra
    groups = {}
    for t in terms:
        groups.setdefault(find(t), []).append(t)
    return groups

shared = mod[mod["module_type"] == "Shared"].copy()
_score_of = dict(zip(shared["pathway"], shared["mean_score"]))
_dir_of = dict(zip(shared["pathway"], shared["direction"]))
_n_tis_all = dict(zip(shared["pathway"], shared["n_tissues"]))

# ---------- 第一步 + 第二步：构建「通路 -> 组」映射 ----------
# 组用 group_id 标识；同组内的通路合并为一条主通路。
group_of = {}          # pathway -> group_id
group_members = {}     # group_id -> [pathways]

# 第一步：同方向内 Jaccard 归并
_gid = 0
for direction, grp in shared.groupby("direction"):
    terms = [t for t in grp["pathway"] if t in genesets]
    if not terms:
        continue
    for root, mem in _merge_by_similarity(terms, _jaccard, JACCARD_THR).items():
        gid = f"J{_gid:04d}"
        _gid += 1
        group_members[gid] = list(mem)
        for t in mem:
            group_of[t] = gid

# 第二步：强制家族 —— 把家族成员的组整体并入一个家族组
forced_merges = []     # (family_name, [被并入的 group_id])
for fam_name, members in FORCED_FAMILIES.items():
    present = [t for t in members if t in group_of]
    if len(present) < 2:
        continue
    gids = []
    for t in present:
        g = group_of[t]
        if g not in gids:
            gids.append(g)
    # 新建家族组，吸收所有涉及组的成员
    fam_gid = f"FAM|{fam_name}"
    absorbed = []
    for g in gids:
        absorbed.extend(group_members.pop(g, []))
    absorbed = sorted(set(absorbed))
    group_members[fam_gid] = absorbed
    for t in absorbed:
        group_of[t] = fam_gid
    forced_merges.append({
        "family": fam_name,
        "family_id": fam_gid,
        "n_members": len(absorbed),
        "n_groups_absorbed": len(gids),
        "members": "|".join(absorbed),
        "directions": "|".join(sorted({_dir_of.get(t, "?") for t in absorbed})),
    })

# ---------- 汇总为 families 表 ----------
merge_rows, member_rows2 = [], []
for gid, mem in group_members.items():
    if not mem:
        continue
    rep = max(mem, key=lambda t: abs(_score_of.get(t, 0.0)))
    dirs = sorted({_dir_of.get(t, "?") for t in mem})
    direction = dirs[0] if len(dirs) == 1 else "mixed"
    merge_rows.append({
        "family_id": gid,
        # family 列 = 家族名（FAM|XXX -> XXX）；自动组为空
        "family": gid.split("|", 1)[1] if gid.startswith("FAM|") else "",
        "direction": direction,
        "representative": rep,
        "n_pathways": len(mem),
        "max_n_tissues": int(max(_n_tis_all.get(t, 0) for t in mem)),
        "members": "|".join(sorted(mem)),
    })
    for t in mem:
        member_rows2.append({
            "pathway": t,
            "family_id": gid,
            "representative": rep,
            "direction": _dir_of.get(t, "?"),
            "is_representative": t == rep,
            "jaccard_to_rep": 1.0 if t == rep else _jaccard(t, rep),
        })

families = pd.DataFrame(merge_rows).sort_values(
    ["family_id"], ascending=[True]).reset_index(drop=True)
pathway_to_family = pd.DataFrame(member_rows2)

n_before, n_after = len(shared), len(families)
print(f"Shared 模块归并: {n_before} -> {n_after}  "
      f"(压缩 {100 * (1 - n_after / n_before):.1f}%)")
print(f"  第一步(Jaccard 同方向) + 第二步(强制家族跨方向)")

if forced_merges:
    print("\n===== 强制家族归并明细 =====")
    for fm in forced_merges:
        print(f"  [{fm['family']}] {fm['n_members']} 条通路 "
              f"(吸收 {fm['n_groups_absorbed']} 个原组, 方向={fm['directions']})")
        for t in fm["members"].split("|"):
            mark = " *" if t == families.set_index("family_id").loc[fm["family_id"], "representative"] else "  "
            print(f"    {mark} {t}")

# ---------- 交叉验证：net_corr ----------
verify_recs = []
for direction, grp in shared.groupby("direction"):
    terms = [t for t in grp["pathway"] if t in genesets]
    if len(terms) < 2:
        continue
    net_parts = [net[net["source"].isin(terms)] for net in nets.values()]
    net_parts = [p for p in net_parts if len(p)]
    if not net_parts:
        continue
    sub_net = pd.concat(net_parts, ignore_index=True)
    try:
        corr_df = dc.pp.net_corr(sub_net, tmin=5)
    except Exception as e:
        print(f"  [verify] net_corr 跳过 direction={direction}: {e}")
        continue
    fam_of = dict(zip(pathway_to_family["pathway"], pathway_to_family["family_id"]))
    for _, r in corr_df.iterrows():
        a, b = r["source_a"], r["source_b"]
        verify_recs.append({
            "direction": direction, "source_a": a, "source_b": b,
            "net_corr": r["corr"], "net_corr_padj": r["padj"],
            "jaccard": _jaccard(a, b),
            "same_family": fam_of.get(a) is not None and fam_of.get(a) == fam_of.get(b),
        })
verify = pd.DataFrame(verify_recs)

if len(verify):
    agree = verify[verify["same_family"]]
    print(f"\n交叉验证（direction 内全部通路对 {len(verify):,} 对）:")
    print(f"  被判为同族: {len(agree)} 对")
    if len(agree):
        hi = (agree["net_corr"] >= VERIFY_CORR).mean()
        print(f"  其中 net_corr >= {VERIFY_CORR} 的比例: {hi:.1%}  "
              f"(中位 net_corr = {agree['net_corr'].median():.3f})")

print("\n===== 归并后主通路（>=2 条成员） =====")
_disp = families[families["n_pathways"] >= 2].copy()
_disp["representative"] = _disp["representative"].str.replace(
    r"^(GOBP_|REACTOME_|HALLMARK_)", "", regex=True).str.slice(0, 52)
print(_disp[["family", "direction", "representative", "n_pathways"]].to_string(index=False))

# ---------- 供下游作图：主通路代表 ----------
# ⚠️ 关键：代表通路的**展示用打分谱**取「组内成员在 8 节点上的均值」，
#    而非仅代表通路自身的打分 —— 归并后应体现整个家族的共同信号。
_node_scores = score_mat.T   # 通路 × 节点
_node_padj = padj_mat.T
rep_rows = []
for _, r in families.iterrows():
    mem = r["members"].split("|")
    # 组内均值打分谱（只对存在数据的成员求均值）
    sub_s = _node_scores.loc[[t for t in mem if t in _node_scores.index]]
    sub_p = _node_padj.loc[[t for t in mem if t in _node_padj.index]]
    rep_rows.append({
        "pathway": r["representative"],
        "family_id": r["family_id"],
        "family": r["family"],
        "direction": r["direction"],
        "n_tissues": r["max_n_tissues"],
        "n_pathways": r["n_pathways"],
        "mean_score": _score_of.get(r["representative"], np.nan),
    })
shared_reps = pd.DataFrame(rep_rows)
# 组均值打分谱（供热图使用）
family_score = {}
family_padj = {}
for _, r in families.iterrows():
    mem = [t for t in r["members"].split("|") if t in _node_scores.index]
    family_score[r["representative"]] = _node_scores.loc[mem].mean(axis=0)
    # 组内 p 值取最小（最保守：任一成员显著即认为该节点显著）
    family_padj[r["representative"]] = _node_padj.loc[mem].min(axis=0)
family_score = pd.DataFrame(family_score).T            # 代表 × 节点
family_padj = pd.DataFrame(family_padj).T
print(f"\nshared_reps: {len(shared_reps)} 条代表通路（供 Fig4B 系列作图）")
print(f"family_score matrix: {family_score.shape}（组内均值打分谱）")

Shared 模块归并: 118 -> 42  (压缩 64.4%)
  第一步(Jaccard 同方向) + 第二步(强制家族跨方向)

===== 强制家族归并明细 =====
  [OXPHOS] 9 条通路 (吸收 5 个原组, 方向=up_in_old)
       GOBP_AEROBIC_ELECTRON_TRANSPORT_CHAIN
       GOBP_ATP_SYNTHESIS_COUPLED_ELECTRON_TRANSPORT
       GOBP_ELECTRON_TRANSPORT_CHAIN
       GOBP_OXIDATIVE_PHOSPHORYLATION
       GOBP_PROTON_MOTIVE_FORCE_DRIVEN_ATP_SYNTHESIS
       HALLMARK_OXIDATIVE_PHOSPHORYLATION
       REACTOME_AEROBIC_RESPIRATION_AND_RESPIRATORY_ELECTRON_TRANSPORT
       REACTOME_COMPLEX_I_BIOGENESIS
     * REACTOME_RESPIRATORY_ELECTRON_TRANSPORT
  [COMPLEMENT] 3 条通路 (吸收 3 个原组, 方向=up_in_old)
     * GOBP_COMPLEMENT_ACTIVATION
       GOBP_REGULATION_OF_COMPLEMENT_ACTIVATION
       REACTOME_COMPLEMENT_CASCADE
  [STEROID_METABOLISM] 3 条通路 (吸收 3 个原组, 方向=down_in_old)
     * GOBP_STEROID_METABOLIC_PROCESS
       GOBP_STEROL_BIOSYNTHETIC_PROCESS
       REACTOME_METABOLISM_OF_STEROIDS
  [HEME_PORPHYRIN] 7 条通路 (吸收 2 个原组, 方向=up_in_old)
       GOBP_HEME_BIOSYNTHETIC_PROCESS
       GOBP_HEME_MET


交叉验证（direction 内全部通路对 6,126 对）:
  被判为同族: 424 对
  其中 net_corr >= 0.5 的比例: 44.8%  (中位 net_corr = 0.465)

===== 归并后主通路（>=2 条成员） =====
                  family   direction                                       representative  n_pathways
    ANTIGEN_PRESENTATION   up_in_old ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EXOGENOUS_PEP           3
       BACTERIAL_DEFENSE   up_in_old ANTIMICROBIAL_HUMORAL_IMMUNE_RESPONSE_MEDIATED_BY_AN           6
CHEMICAL_STRESS_RESPONSE   up_in_old                          RESPONSE_TO_TOXIC_SUBSTANCE           2
              COMPLEMENT   up_in_old                                COMPLEMENT_ACTIVATION           3
          HEME_PORPHYRIN   up_in_old                                      HEME_METABOLISM           7
   INFLAMMATORY_RESPONSE   up_in_old                  REGULATION_OF_INFLAMMATORY_RESPONSE           3
     LEUKOCYTE_MIGRATION   up_in_old                                  LEUKOCYTE_MIGRATION          17
      MITOSIS_CELL_CYCLE   up_in_old                

In [11]:
# 【完整原始结果】——逐通路打分与模块分类全部落盘，不做任何裁剪或过滤
cube.to_csv(OUT_DIR / "pathway_cube_tissue_celltype_pathway.tsv", sep="	", index=False)
score_mat.to_csv(OUT_DIR / "pathway_score_matrix.tsv", sep="	")
padj_mat.to_csv(OUT_DIR / "pathway_padj_matrix.tsv", sep="	")
score_mat.T.rename_axis("Term").to_csv(OUT_DIR / "pathway_nes_matrix.tsv", sep="	")
padj_mat.T.rename_axis("Term").to_csv(OUT_DIR / "pathway_fdr_matrix.tsv", sep="	")
mod.to_csv(OUT_DIR / "module_summary.tsv", sep="	", index=False)

# 【去冗余层 1：打分谱聚类】（Cell 9）——集合 × 方向内，8 节点打分谱层次聚类
themes.to_csv(OUT_DIR / "pathway_themes.tsv", sep="	", index=False)
themes_merged.to_csv(OUT_DIR / "pathway_themes_merged.tsv", sep="	", index=False)
pathway_to_theme.to_csv(OUT_DIR / "pathway_to_theme.tsv", sep="	", index=False)

# 【去冗余层 2：基因集重叠 + 强制家族】（Cell 10）
# Jaccard 同方向归并 + 人工审核家族跨方向合并（如 OXPHOS）
families.to_csv(OUT_DIR / "pathway_families.tsv", sep="	", index=False)
pathway_to_family.to_csv(OUT_DIR / "pathway_to_family.tsv", sep="	", index=False)
shared_reps.to_csv(OUT_DIR / "shared_module_representatives.tsv", sep="	", index=False)
family_score.to_csv(OUT_DIR / "family_score_matrix.tsv", sep="	")
family_padj.to_csv(OUT_DIR / "family_padj_matrix.tsv", sep="	")
if len(verify):
    verify.to_csv(OUT_DIR / "pathway_merge_verification.tsv", sep="	", index=False)

SAVED = [
    "pathway_cube_tissue_celltype_pathway.tsv", "pathway_score_matrix.tsv",
    "pathway_padj_matrix.tsv", "pathway_nes_matrix.tsv", "pathway_fdr_matrix.tsv",
    "module_summary.tsv", "pathway_themes.tsv", "pathway_themes_merged.tsv",
    "pathway_to_theme.tsv", "pathway_families.tsv", "pathway_to_family.tsv",
    "shared_module_representatives.tsv", "family_score_matrix.tsv",
    "family_padj_matrix.tsv", "pathway_merge_verification.tsv",
]
for fn in SAVED:
    p = OUT_DIR / fn
    if p.exists():
        print(f"saved: {fn}")


saved: pathway_cube_tissue_celltype_pathway.tsv
saved: pathway_score_matrix.tsv
saved: pathway_padj_matrix.tsv
saved: pathway_nes_matrix.tsv
saved: pathway_fdr_matrix.tsv
saved: module_summary.tsv
saved: pathway_themes.tsv
saved: pathway_themes_merged.tsv
saved: pathway_to_theme.tsv
saved: pathway_families.tsv
saved: pathway_to_family.tsv
saved: shared_module_representatives.tsv
saved: family_score_matrix.tsv
saved: family_padj_matrix.tsv
saved: pathway_merge_verification.tsv


In [12]:
# ===================== 10. 可视化（Fig4：通路富集与跨组织共享/特异模块） =====================
# 论文图表清单中 Figure 4 =「通路富集与跨组织共享/特异模块」，
# 注意：本图节点为 8 个「组织×细胞类型」，统计单位是背后的独立小鼠，
#      不得以节点数冒充样本量（计划书：不得用细胞点数制造虚假显著性）。
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

FIG_DIR = ROOT / "figures" / "main"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "DejaVu Sans"   # 避免中文缺字；图中标签统一用英文
plt.rcParams["axes.unicode_minus"] = False

RED, BLUE, AMBER, GREY = "#C0392B", "#2E5C9A", "#E8A33D", "#9AA0A6"
# 红 = up_in_old（年龄上调），蓝 = down_in_old（年龄下调）

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"saved: {FIG_DIR / (name + '.pdf')}")

def short(name, n=44):
    """去掉集合前缀并把下划线换成空格，便于作图标签。"""
    for pre in ("GOBP_", "REACTOME_", "HALLMARK_"):
        if name.startswith(pre):
            name = name[len(pre):]
            break
    return name.replace("_", " ").title()[:n]

print(f"figure output dir: {FIG_DIR}")

figure output dir: ..\figures\main


In [ ]:
# ---------- Fig4-A：模块按「覆盖组织数」的分层分布 ----------
by_n = mod["n_tissues"].value_counts().sort_index()
SIG = 4 * SHARED_FRAC   # Shared 判定门槛（70% × 4 = 2.8）

fig, ax = plt.subplots(figsize=(6.2, 4.0))
colors_n = [BLUE if n == 1 else (AMBER if n < SIG else RED) for n in by_n.index]
bars = ax.bar(by_n.index.astype(str), by_n.values,
              color=colors_n, edgecolor="white", width=0.62)
for b, v in zip(bars, by_n.values):
    ax.text(b.get_x() + b.get_width() / 2, v + max(by_n.values) * 0.02,
            int(v), ha="center", fontsize=10, color="#333333")

ax.set_xlabel("Number of tissues where the module is active")
ax.set_ylabel("Number of modules")
ax.set_title("How many tissues each module spans", fontsize=11)
ax.set_ylim(0, max(by_n.values) * 1.16)

# 标注三类模块的含义（通过图例，而非另一张柱状图）
from matplotlib.patches import Patch
handles = [Patch(facecolor=BLUE, label="Specific (1 tissue)"),
           Patch(facecolor=AMBER, label="Partial (2 tissues)"),
           Patch(facecolor=RED, label="Shared (>= 3 tissues)")]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="upper right")

save(fig, "Fig4A_module_overview")

saved: ..\figures\main\Fig4A_module_overview.pdf


In [ ]:
# ---------- Fig4-B：Shared 主通路 × 节点 热图（核心面板） ----------

SIG_FDR_FIG = 0.05    # 显著性阈值：FDR>=0.05 的格留白

# ---------- 功能分组（人工指定） ----------
FUNC_GROUPS = [
    ("Protein synthesis", [
        "REACTOME_NONSENSE_MEDIATED_DECAY_NMD_INDEPENDENT_OF_THE_EXON_JUNCTION_COMPLEX_EJC",
        "REACTOME_TRANSLATION",
        "REACTOME_MAJOR_PATHWAY_OF_RRNA_PROCESSING_IN_THE_NUCLEOLUS_AND_CYTOSOL",
        "GOBP_TRNA_METABOLIC_PROCESS",
    ]),
    ("Redox homeostasis & cellular respiration", [
        "GOBP_RESPONSE_TO_TOXIC_SUBSTANCE",
        "HALLMARK_HEME_METABOLISM",
        "REACTOME_RESPIRATORY_ELECTRON_TRANSPORT",
        "REACTOME_DETOXIFICATION_OF_REACTIVE_OXYGEN_SPECIES",
        "REACTOME_ROS_AND_RNS_PRODUCTION_IN_PHAGOCYTES",
        "HALLMARK_HYPOXIA",
        "GOBP_RESPIRATORY_BURST",
    ]),
    ("Anti-infection & innate immunity", [
        "GOBP_COMPLEMENT_ACTIVATION",
        "GOBP_ANTIMICROBIAL_HUMORAL_IMMUNE_RESPONSE_MEDIATED_BY_ANTIMICROBIAL_PEPTIDE",
        "GOBP_REGULATION_OF_INFLAMMATORY_RESPONSE",
        "HALLMARK_IL6_JAK_STAT3_SIGNALING",
        "GOBP_MYELOID_LEUKOCYTE_ACTIVATION",
        "GOBP_LEUKOCYTE_MIGRATION",
        "GOBP_NEUROINFLAMMATORY_RESPONSE",
        "REACTOME_G_ALPHA_I_SIGNALLING_EVENTS",
        "GOBP_RESPONSE_TO_FUNGUS",
        "REACTOME_C_TYPE_LECTIN_RECEPTORS_CLRS",
        "HALLMARK_INTERFERON_GAMMA_RESPONSE",
        "HALLMARK_IL2_STAT5_SIGNALING",
        "REACTOME_INTERLEUKIN_1_SIGNALING",
        "HALLMARK_APICAL_JUNCTION",
    ]),
    ("Cell death & necrosis", [
        "GOBP_CELL_KILLING",
        "GOBP_DISRUPTION_OF_ANATOMICAL_STRUCTURE_IN_ANOTHER_ORGANISM",
        "GOBP_INFLAMMATORY_CELL_APOPTOTIC_PROCESS",
        "HALLMARK_P53_PATHWAY",
    ]),
    ("Protein & antigen processing and presentation", [
        "GOBP_EXOCYTOSIS",
        "REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION",
        "GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EXOGENOUS_PEPTIDE_ANTIGEN",
        "GOBP_ESTABLISHMENT_OF_PROTEIN_LOCALIZATION_TO_ENDOPLASMIC_RETICULUM",
        "GOBP_PROTEIN_EXIT_FROM_ENDOPLASMIC_RETICULUM",
    ]),
    ("Cell cycle regulation", [
        "REACTOME_MITOTIC_METAPHASE_AND_ANAPHASE",
        "GOBP_POSITIVE_REGULATION_OF_CHROMOSOME_SEPARATION",
        "HALLMARK_MYC_TARGETS_V1",
    ]),
    ("Adaptive immunity", [
        "REACTOME_CO_INHIBITION_BY_PD_1",
        "HALLMARK_ALLOGRAFT_REJECTION",
        "GOBP_REGULATION_OF_T_CELL_ACTIVATION",
    ]),
    ("Lipid metabolism", [
        "GOBP_STEROID_METABOLIC_PROCESS",
        "GOBP_LIPID_CATABOLIC_PROCESS",
    ]),
]

sr = shared_reps.copy()
sr["_abs"] = sr["mean_score"].abs()
sr = sr[sr["pathway"].isin(family_score.index)].reset_index(drop=True)

# --- 按人工功能组排列行；组内按 n_tissues↓ -> direction -> |score|↓ ---
_all_reps = set(sr["pathway"])
_assigned = [t for _, mem in FUNC_GROUPS for t in mem]
_unassigned = [t for t in sr["pathway"] if t not in set(_assigned)]
if _unassigned:
    print(f"  [warn] 未被功能组覆盖的代表通路: {_unassigned}")
_missing = [t for t in _assigned if t not in _all_reps]
if _missing:
    print(f"  [warn] 功能组名单里不在 shared_reps 中的名字: {_missing}")

rows_ord, cluster_of, cluster_bounds = [], {}, []
for gi, (gname, members) in enumerate(FUNC_GROUPS):
    present = [t for t in members if t in _all_reps]
    if not present:
        print(f"  [warn] 功能组「{gname}」无有效成员，跳过")
        continue
    m = sr[sr["pathway"].isin(present)].copy()
    m["_dir"] = m["direction"].map({"up_in_old": 0, "mixed": 1, "down_in_old": 2}).fillna(1)
    m = m.sort_values(["_dir", "n_tissues", "_abs"], ascending=[True, False, False])
    start = len(rows_ord)
    rows_ord += list(m["pathway"])
    for t in m["pathway"]:
        cluster_of[t] = gi
    cluster_bounds.append((start, len(rows_ord), gi))

sr = sr.set_index("pathway").loc[rows_ord].reset_index()
sel_terms = list(sr["pathway"])
n_rows = len(sel_terms)
_n_groups = len(cluster_bounds)
print(f"Fig4B(主图) 绘制主通路数: {n_rows}  |  人工功能组 {_n_groups} 个")
GROUP_NAME = {gi: gname for gi, (gname, _) in enumerate(FUNC_GROUPS)}
for (a, b, gi) in cluster_bounds:
    nm = GROUP_NAME[gi]
    mx = int(sr.iloc[a:b]["n_tissues"].max())
    print(f"  [{nm}] {b - a:2d} 行  (max n_tis={mx})  起始: {short(sr.iloc[a]['pathway'])[:40]}")

# 用组均值打分谱作图（不是单条代表通路的打分）
mat_b = family_score.loc[sel_terms]
padj_b = family_padj.loc[sel_terms]

mat_plot = mat_b.where(padj_b < SIG_FDR_FIG)
n_blank = int(mat_plot.isna().sum().sum())
print(f"  非显著格（置白）: {n_blank} / {mat_plot.size} ({n_blank / mat_plot.size:.1%})")

n_tis_map = dict(zip(sr["pathway"], sr["n_tissues"]))
n_mem_map = dict(zip(sr["pathway"], sr["n_pathways"]))

def _lab(t):
    base = short(t)
    n = n_tis_map.get(t, "?")
    k = n_mem_map.get(t, 1)
    return f"{base}  [{n}]" if k == 1 else f"{base}  [{n}] ({k})"

vmax = float(np.nanmax(np.abs(mat_plot.values))) or 1.0
fig, ax = plt.subplots(figsize=(8.4, max(9.0, n_rows * 0.245)))
sns.heatmap(mat_plot, cmap="RdBu_r",
            norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
            cbar_kws={"label": "ULM score", "shrink": 0.22, "pad": 0.02},
            linewidths=0.4, linecolor="white", ax=ax)

for i in range(mat_plot.shape[0]):
    for j in range(mat_plot.shape[1]):
        if np.isfinite(padj_b.values[i, j]) and padj_b.values[i, j] < SIG_FDR_FIG:
            ax.text(j + 0.5, i + 0.5, "*", ha="center", va="center",
                    fontsize=7, color="black")

# 功能组分界线（只画内部边界，不画上下外沿）
for (a, b, lb) in cluster_bounds[:-1]:
    ax.axhline(b, color="#2C2C2A", linewidth=1.5)

ax.set_title("Shared modules across tissue x cell type\n"
             "(rows grouped into curated functional categories, sorted by #tissues within each group;\n"
             "[n] = #tissues, (k) = #pathways merged)",
             fontsize=10.5, pad=10)
ax.set_xticks(np.arange(mat_plot.shape[1]) + 0.5)
ax.set_xticklabels([c.replace("__", " | ") for c in mat_plot.columns],
                   rotation=30, ha="right", fontsize=9)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_yticklabels([_lab(t) for t in mat_plot.index], fontsize=5.6)
ax.set_xlabel(""); ax.set_ylabel("")
fig.subplots_adjust(left=0.47, bottom=0.20)

# 功能组标签：用轴坐标（transAxes）定位，x<0 且随 left 边距自适应；
# ⚠️ 必须在 set_yticks/subplots_adjust **之后** 再算，否则 y 换算会失配。
# 功能组标签：放在**图最左侧的独立列**（figure 坐标，左对齐）。
# ⚠️ 用 ax.transAxes 做「轴内相对坐标 -> figure 坐标」的标准换算，
#    切勿手工拼 ax.get_position()（易把上下方向搞反）。
for (a, b, lb) in cluster_bounds:
    y_ax = 1.0 - (a + b) / 2.0 / n_rows          # 轴内相对 y（0=顶, 1=底）
    y_fig = ax.transAxes.transform((0.0, y_ax))[1]      # -> display
    y_fig = fig.transFigure.inverted().transform((0.0, y_fig))[1]   # -> figure
    _lbl = GROUP_NAME[lb].replace(" & ", " &\n").replace(" and ", " and\n")
    fig.text(0.008, y_fig, _lbl, ha="left", va="center",
             fontsize=6.4, color="#2C2C2A", fontweight="bold",
             linespacing=1.3,
             bbox=dict(boxstyle="round,pad=0.20", facecolor="white",
                       edgecolor="#2C2C2A", linewidth=0.7))

ax.text(0.0, -0.235,
        "* FDR < 0.05   |   blank = FDR >= 0.05   |   [n] = #active tissues, (k) = #merged pathways\n"
        "rows grouped into curated functional categories; within each group sorted by direction then #tissues",
        transform=ax.transAxes, fontsize=7.5, color="#5F5E5A", va="top", ha="left")
save(fig, "Fig4B_pathway_heatmap")

# 落盘行序与簇归属，供下游核对
pd.DataFrame({
    "row_order": np.arange(n_rows),
    "pathway": sel_terms,
    "group_id": [cluster_of[t] for t in sel_terms],
    "group_name": [GROUP_NAME[cluster_of[t]] for t in sel_terms],
    "n_tissues": [n_tis_map[t] for t in sel_terms],
    "n_pathways": [n_mem_map[t] for t in sel_terms],
    "direction": list(sr["direction"]),
}).to_csv(OUT_DIR / "fig4b_row_order.tsv", sep="\t", index=False)
print("saved: fig4b_row_order.tsv")


Fig4B(主图) 绘制主通路数: 42  |  人工功能组 8 个
  [Protein synthesis]  4 行  (max n_tis=4)  起始: Nonsense Mediated Decay Nmd Independent 
  [Redox homeostasis & cellular respiration]  7 行  (max n_tis=4)  起始: Response To Toxic Substance
  [Anti-infection & innate immunity] 14 行  (max n_tis=4)  起始: Antimicrobial Humoral Immune Response Me
  [Cell death & necrosis]  4 行  (max n_tis=4)  起始: Cell Killing
  [Protein & antigen processing and presentation]  5 行  (max n_tis=3)  起始: Antigen Processing And Presentation Of E
  [Cell cycle regulation]  3 行  (max n_tis=3)  起始: Mitotic Metaphase And Anaphase
  [Adaptive immunity]  3 行  (max n_tis=3)  起始: Allograft Rejection
  [Lipid metabolism]  2 行  (max n_tis=3)  起始: Steroid Metabolic Process
  非显著格（置白）: 145 / 336 (43.2%)


saved: ..\figures\main\Fig4B_pathway_heatmap.pdf
saved: fig4b_row_order.tsv


In [ ]:
# ---------- Fig4B-1 / B-2：按「细胞类型」拆分的热图 ----------

CT_LABEL = {"lymphoid": "Lymphoid", "myeloid/macrophage": "Myeloid / macrophage"}

# 主图的簇归属（通路 -> 簇编号）与簇顺序
_cl_of = dict(zip(sel_terms, [cluster_of[t] for t in sel_terms]))
_cl_order = [lb for _, _, lb in cluster_bounds]

def _draw_cluster_lines(ax, kept, fig, n_rows, x_right):
    """在拆分图上按主图簇顺序画分隔线 + 功能名标签（仅对保留的行）。"""
    # 统计每个簇在 kept 中占的连续段
    seq = [_cl_of[t] for t in kept]
    bounds, start = [], 0
    for i in range(1, len(seq) + 1):
        if i == len(seq) or seq[i] != seq[start]:
            bounds.append((start, i, seq[start]))
            start = i
    for (a, b, lb) in bounds[:-1]:
        ax.axhline(b, color="#2C2C2A", linewidth=1.2)
    _px = ax.transData.transform((0, 0))[0]
    _px = fig.transFigure.inverted().transform((_px, 0))[0]
    for (a, b, lb) in bounds:
        y_fig = ax.transData.transform((0, (a + b) / 2.0))[1]
        y_fig = fig.transFigure.inverted().transform((0, y_fig))[1]
        fig.text(_px - 0.52, y_fig, GROUP_NAME.get(lb, f"G{lb}"),
                 ha="center", va="center", fontsize=5.6, color="#2C2C2A",
                 fontweight="bold", linespacing=1.2,
                 bbox=dict(boxstyle="round,pad=0.18", facecolor="white",
                           edgecolor="#2C2C2A", linewidth=0.5))
    return bounds

for ct_key, ct_name in CT_LABEL.items():
    cols = [n for n in family_score.columns if n.split("__")[1] == ct_key]
    cols = sorted(cols, key=lambda n: ["Kidney", "Liver", "Lung", "spleen/marrow"]
                  .index(n.split("__")[0]))

    sub = family_score.loc[sel_terms, cols]
    psub = family_padj.loc[sel_terms, cols]

    row_sig = (psub < SIG_FDR_FIG).any(axis=1)
    dropped = [t for t in sub.index if not row_sig.get(t, False)]
    kept = [t for t in sub.index if row_sig.get(t, False)]
    sub, psub = sub.loc[row_sig], psub.loc[row_sig]
    mat_plot = sub.where(psub < SIG_FDR_FIG)

    vmax = float(np.nanmax(np.abs(mat_plot.values))) or 1.0
    n_rows = mat_plot.shape[0]
    fig, ax = plt.subplots(figsize=(4.6, max(6.0, n_rows * 0.235)))
    sns.heatmap(mat_plot, cmap="RdBu_r",
                norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
                cbar_kws={"label": "ULM score", "shrink": 0.18, "pad": 0.03},
                linewidths=0.4, linecolor="white", ax=ax)
    for i in range(mat_plot.shape[0]):
        for j in range(mat_plot.shape[1]):
            if np.isfinite(psub.values[i, j]) and psub.values[i, j] < SIG_FDR_FIG:
                ax.text(j + 0.5, i + 0.5, "*", ha="center", va="center",
                        fontsize=6.5, color="black")

    _b = _draw_cluster_lines(ax, kept, fig, n_rows, len(cols))

    ax.set_title(f"Shared modules in {ct_name}\n({len(cols)} tissues; blank = FDR >= 0.05)",
                 fontsize=10, pad=8)
    ax.set_xticks(np.arange(len(cols)) + 0.5)
    ax.set_xticklabels([c.split("__")[0] for c in cols],
                       rotation=40, ha="right", fontsize=9)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels([_lab(t) for t in mat_plot.index], fontsize=5.6)
    ax.set_xlabel(""); ax.set_ylabel("")
    fig.subplots_adjust(left=0.56, bottom=0.16)
    tag = "lymphoid" if ct_key == "lymphoid" else "myeloid"
    save(fig, f"Fig4B_byCellType_{tag}")
    print(f"  [{ct_name}] 保留 {n_rows} 行 (删除全非显著 {len(dropped)} 行), 列={len(cols)}, 簇段={len(_b)}")


saved: ..\figures\main\Fig4B_byCellType_lymphoid.pdf
  [Lymphoid] 保留 37 行 (删除全非显著 5 行), 列=4, 簇段=8


saved: ..\figures\main\Fig4B_byCellType_myeloid.pdf
  [Myeloid / macrophage] 保留 40 行 (删除全非显著 2 行), 列=4, 簇段=8


In [ ]:
# ---------- Fig4B-3 ~ B-6：按「组织」拆分的热图 ----------

ORGAN_ORDER = ["Kidney", "Liver", "Lung", "spleen/marrow"]
CT_SHORT = {"lymphoid": "Lymphoid", "myeloid/macrophage": "Myeloid"}

for tis in ORGAN_ORDER:
    cols = [n for n in family_score.columns if n.split("__")[0] == tis]
    cols = sorted(cols, key=lambda n: 0 if n.split("__")[1] == "lymphoid" else 1)

    sub = family_score.loc[sel_terms, cols]
    psub = family_padj.loc[sel_terms, cols]

    row_sig = (psub < SIG_FDR_FIG).any(axis=1)
    dropped = [t for t in sub.index if not row_sig.get(t, False)]
    kept = [t for t in sub.index if row_sig.get(t, False)]
    sub, psub = sub.loc[row_sig], psub.loc[row_sig]
    mat_plot = sub.where(psub < SIG_FDR_FIG)

    vmax = float(np.nanmax(np.abs(mat_plot.values))) or 1.0
    n_rows = mat_plot.shape[0]
    fig, ax = plt.subplots(figsize=(4.0, max(6.0, n_rows * 0.235)))
    sns.heatmap(mat_plot, cmap="RdBu_r",
                norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
                cbar_kws={"label": "ULM score", "shrink": 0.18, "pad": 0.04},
                linewidths=0.4, linecolor="white", ax=ax)
    for i in range(mat_plot.shape[0]):
        for j in range(mat_plot.shape[1]):
            if np.isfinite(psub.values[i, j]) and psub.values[i, j] < SIG_FDR_FIG:
                ax.text(j + 0.5, i + 0.5, "*", ha="center", va="center",
                        fontsize=6.5, color="black")

    _b = _draw_cluster_lines(ax, kept, fig, n_rows, len(cols))

    ax.set_title(f"Shared modules in {tis}\n(lymphoid vs myeloid; blank = FDR >= 0.05)",
                 fontsize=10, pad=8)
    ax.set_xticks(np.arange(len(cols)) + 0.5)
    ax.set_xticklabels([CT_SHORT[c.split("__")[1]] for c in cols],
                       rotation=40, ha="right", fontsize=9)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels([_lab(t) for t in mat_plot.index], fontsize=5.6)
    ax.set_xlabel(""); ax.set_ylabel("")
    fig.subplots_adjust(left=0.56, bottom=0.16)
    tag = tis.replace("/", "_")
    save(fig, f"Fig4B_byOrgan_{tag}")
    print(f"  [{tis}] 保留 {n_rows} 行 (删除全非显著 {len(dropped)} 行), 列={len(cols)}, 簇段={len(_b)}")


saved: ..\figures\main\Fig4B_byOrgan_Kidney.pdf
  [Kidney] 保留 37 行 (删除全非显著 5 行), 列=2, 簇段=8


saved: ..\figures\main\Fig4B_byOrgan_Liver.pdf
  [Liver] 保留 41 行 (删除全非显著 1 行), 列=2, 簇段=8


saved: ..\figures\main\Fig4B_byOrgan_Lung.pdf
  [Lung] 保留 29 行 (删除全非显著 13 行), 列=2, 簇段=8


saved: ..\figures\main\Fig4B_byOrgan_spleen_marrow.pdf
  [spleen/marrow] 保留 38 行 (删除全非显著 4 行), 列=2, 簇段=8


In [ ]:
# ---------- Fig4-C：去冗余压缩比（三层） ----------
comp = (themes.groupby(["collection", "direction"])
        .agg(n_pathways=("n_pathways", "sum"), n_themes=("theme_id", "count"))
        .reset_index())
comp["label"] = (comp["collection"].str.replace("_", " ").str.replace("pathways", "").str.strip()
                 + " | " + comp["direction"].str.replace("_in_old", ""))
comp = comp.sort_values("n_pathways", ascending=True)

fig, ax = plt.subplots(figsize=(8.4, 4.2))
y = np.arange(len(comp))
ax.barh(y + 0.19, comp["n_pathways"], height=0.36, color=GREY, label="raw pathways")
ax.barh(y - 0.19, comp["n_themes"], height=0.36, color=RED, label="merged themes (layer 1)")
for i, (raw, th) in enumerate(zip(comp["n_pathways"], comp["n_themes"])):
    ax.text(raw + 8, i + 0.19, f"{raw}", va="center", fontsize=8, color="#444441")
    ax.text(th + 8, i - 0.19, f"{th}", va="center", fontsize=8, color=RED)
    ax.text(raw * 0.5, i, f"{raw / th:.0f}x", va="center", ha="center",
            fontsize=8.5, style="italic", color="#333333")

# --- 层2：Shared 模块 118 -> 家族 63（单独一组，放在最下方） ---
n_shared = int((mod["module_type"] == "Shared").sum())
n_family = int(len(families))
i2 = -1.35
ax.barh([i2 + 0.19], [n_shared], height=0.36, color=GREY)
ax.barh([i2 - 0.19], [n_family], height=0.36, color=RED)
ax.text(n_shared + 8, i2 + 0.19, f"{n_shared}", va="center", fontsize=8, color="#444441")
ax.text(n_family + 8, i2 - 0.19, f"{n_family}", va="center", fontsize=8, color=RED)
ax.text(n_shared * 0.5, i2, f"{n_shared / n_family:.1f}x", va="center", ha="center",
        fontsize=8.5, style="italic", color="#333333")

labels = list(comp["label"]) + ["Shared modules -> families\n(layer 2, cross-collection)"]
ticks = list(y) + [i2]
ax.set_yticks(ticks)
ax.set_yticklabels(labels, fontsize=8.5)
ax.set_xlabel("Number of pathways / themes / families")
ax.set_title("Redundancy removal: two layers of compression", fontsize=12)
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(0, max(comp["n_pathways"].max(), n_shared) * 1.2)
save(fig, "Fig4C_compression")


saved: ..\figures\main\Fig4C_compression.pdf


In [ ]:
# ---------- 补充图 S1：主题树状图（可视化去冗余原理） ----------
from scipy.cluster.hierarchy import dendrogram
from scipy.spatial.distance import squareform

big = (themes.groupby(["collection", "direction"])["n_pathways"].sum()
       .sort_values(ascending=False))
big_coll, big_dir = big.index[0]
terms_s1 = [t for t in mod[(mod["collection"] == big_coll) &
                           (mod["direction"] == big_dir)]["pathway"]
            if t in score_mat.columns]

sub_s1 = score_mat[terms_s1].T.dropna(axis=0, how="any")
corr_s1 = sub_s1.T.corr().fillna(0)
dist_s1 = 1 - corr_s1
np.fill_diagonal(dist_s1.values, 0)
z_s1 = linkage(squareform(dist_s1.values, checks=False), method="average")

fig, ax = plt.subplots(figsize=(9, 4.5))
dendrogram(z_s1, no_labels=True, color_threshold=1 - CLUSTER_R, ax=ax)
ax.axhline(1 - CLUSTER_R, color=RED, linestyle="--", linewidth=1.4,
           label=f"cut at r > {CLUSTER_R}  (distance = {1 - CLUSTER_R:.2f})")
ax.set_ylabel("Distance (1 - Pearson r)")
ax.set_xlabel(f"{len(terms_s1)} pathways -> "
              f"{themes[(themes.collection == big_coll) & (themes.direction == big_dir)].shape[0]} themes")
ax.set_title(f"Hierarchical clustering of redundant pathways  ({big_coll} | {big_dir})",
             fontsize=11)
ax.legend(frameon=False)
save(fig, "SuppS1_theme_dendrogram")

saved: ..\figures\main\SuppS1_theme_dendrogram.pdf
